# 317 — Layer 3 Assessment Cleanup

Removes **Assessment**, **Finding**, and **ReasoningStep** nodes (and all their
relationships) written to Neo4j by the agent pipeline.

Run this notebook to reset Layer 3 back to a clean state without touching
Layer 1 (entities) or Layer 2 (regulatory graph).

In [1]:
%run 311_agent_setup.ipynb

/Users/emilpastor/opt/miniconda3/envs/loanguard-ai/lib/python3.11/site-packages/nbformat/__init__.py:96: MissingIDFieldWarning: Cell is missing an id field, this will become a hard error in future nbformat versions. You may want to use `normalize()` on your notebooks before validations (available since nbformat 5.1.4). Previous versions of nbformat are fixing this issue transparently, and will stop doing so in the future.
  validate(nb)


Environment loaded.


2026-03-10 20:37:31,881 [INFO] src.graph.connection: Connected to Neo4j at neo4j+s://44f9022d.databases.neo4j.io


Connected to Neo4j.

Node counts in graph:
  {'labels': {'CommercialSecured': 2, 'Federal': 1, 'Address': 609, 'Residential': 582, 'Corporate': 31, 'Assessment': 13, 'Collateral': 466, 'ResidentialSecured': 464, 'Industry': 14, 'National': 5, 'PersonalTransaction': 610, 'Threshold': 133, 'CorporateCurrent': 6, 'Borrower': 628, 'BusinessTransaction': 175, 'SpecialAdminRegion': 1, 'BankAccount': 791, 'Chunk': 189, 'Section': 101, 'Jurisdiction': 7, 'Commercial': 27, 'Requirement': 219, 'Transaction': 173, 'LoanApplication': 466, 'Individual': 597, 'Finding': 40, 'Regulation': 3, 'ReasoningStep': 61, 'Officer': 19}}
  ✓  Borrower: 628
  ✓  LoanApplication: 466
  ✓  BankAccount: 791
  ✓  Transaction: 173
  ✓  Regulation: 3
  ✓  Section: 101
  ✓  Requirement: 219
  ✓  Threshold: 133
  ✓  Chunk: 189
Anthropic client ready. Model: claude-sonnet-4-6
OpenAI client ready.
Tool registry: 2 Neo4j MCP + 6 FastMCP = 8 total
  read-neo4j-cypher
  write-neo4j-cypher
  traverse_compliance_path
  retrie

## 1. Inspect current Layer 3 state

In [2]:
# Count Layer 3 nodes before cleanup
counts = conn.run_query("""
    MATCH (a:Assessment)
    OPTIONAL MATCH (a)-[:HAS_FINDING]->(f:Finding)
    OPTIONAL MATCH (a)-[:HAS_STEP]->(rs:ReasoningStep)
    RETURN
        count(DISTINCT a)  AS assessments,
        count(DISTINCT f)  AS findings,
        count(DISTINCT rs) AS reasoning_steps
""")
print('Layer 3 nodes before cleanup:')
for row in counts:
    print(f'  Assessments:     {row["assessments"]}')
    print(f'  Findings:        {row["findings"]}')
    print(f'  ReasoningSteps:  {row["reasoning_steps"]}')

Layer 3 nodes before cleanup:
  Assessments:     13
  Findings:        40
  ReasoningSteps:  61


## 2. Preview assessments (optional)

In [3]:
# List assessments that will be deleted
rows = conn.run_query("""
    MATCH (a:Assessment)
    RETURN a.assessment_id AS id,
           a.entity_id     AS entity,
           a.verdict       AS verdict,
           a.agent         AS agent,
           a.created_at    AS created_at
    ORDER BY a.created_at DESC
    LIMIT 50
""")
print(f'{len(rows)} assessment(s) found:')
for r in rows:
    print(f'  {r["id"]}  entity={r["entity"]}  verdict={r["verdict"]}  agent={r["agent"]}')

13 assessment(s) found:
  ASSESS-LOAN-0006-APS-220-2026-03-10-202954  entity=LOAN-0006  verdict=REQUIRES_REVIEW  agent=compliance_agent
  ASSESS-LOAN-0006-APS-112-2026-03-10-202954  entity=LOAN-0006  verdict=COMPLIANT  agent=compliance_agent
  ASSESS-LOAN-0006-APG-223-2026-03-10-202953  entity=LOAN-0006  verdict=REQUIRES_REVIEW  agent=compliance_agent
  ASSESS-LOAN-0006-APG-223-2026-03-10-202715  entity=LOAN-0006  verdict=COMPLIANT  agent=compliance_agent
  ASSESS-LOAN-0006-APG-223-2026-03-10-201927  entity=LOAN-0006  verdict=COMPLIANT  agent=compliance_agent
  ASSESS-LOAN-0002-APG-223-2026-03-10-201749  entity=LOAN-0002  verdict=NON_COMPLIANT  agent=compliance_agent
  ASSESS-LOAN-0006-APG-223-2026-03-10-201608  entity=LOAN-0006  verdict=REQUIRES_REVIEW  agent=compliance_agent
  ASSESS-LOAN-0006-APG-223-2026-03-10-201546  entity=LOAN-0006  verdict=COMPLIANT  agent=compliance_agent
  ASSESS-LOAN-0001-APS-112-2026-03-10-200232  entity=LOAN-0001  verdict=REQUIRES_REVIEW  agent=compliance_

## 3. Delete Layer 3 nodes

> **Warning** — this is irreversible. Run the preview cell above first.

In [4]:
result = conn.run_query("""
    MATCH (a:Assessment)
    OPTIONAL MATCH (a)-[:HAS_FINDING]->(f:Finding)
    OPTIONAL MATCH (a)-[:HAS_STEP]->(rs:ReasoningStep)
    DETACH DELETE a, f, rs
""")
print('Layer 3 nodes deleted.')

Layer 3 nodes deleted.


## 4. Verify cleanup

In [5]:
# Confirm all Layer 3 nodes are gone
after = conn.run_query("""
    MATCH (a:Assessment)
    OPTIONAL MATCH (a)-[:HAS_FINDING]->(f:Finding)
    OPTIONAL MATCH (a)-[:HAS_STEP]->(rs:ReasoningStep)
    RETURN
        count(DISTINCT a)  AS assessments,
        count(DISTINCT f)  AS findings,
        count(DISTINCT rs) AS reasoning_steps
""")
row = after[0]
assert row['assessments'] == 0, 'Assessments still present!'
assert row['findings'] == 0, 'Findings still present!'
assert row['reasoning_steps'] == 0, 'ReasoningSteps still present!'
print('All Layer 3 nodes removed. Graph is clean.')

All Layer 3 nodes removed. Graph is clean.
